# Analyzing a run

Loads a run folder written by the sim logger (`experiments/output/<run-id>/`) and shows what tuning needs: food, time, energy, food sources, per-Folk outcomes and timing. The analysis code lives in `scripts/analyze_run.py`, which can also be run from the command line to write all plots.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts') if Path.cwd().name == 'notebooks' else 'scripts')
import pandas as pd
import matplotlib.pyplot as plt
import analyze_run as ar

pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

## Pick a run
The newest run by default. Set `path` to look at another one.

In [ ]:
path = None  # e.g. Path('../experiments/output/20260920-150658-seed2')
run = ar.Run(path or ar.newest_run())
run

## Total food over time
World stock relative to the start, and food carried by Folk.

In [ ]:
ar.plot_food_over_time(run);
ar.food_over_time(run).tail()

## Calorie reserve
Mean, minimum and maximum reserve per decider over time, as a share of capacity. A reserve near zero means Folk are close to starving.

In [ ]:
ar.plot_reserve(run);
ar.reserve_over_time(run).tail()

## Food by terrain
How full each terrain type is per species. A terrain near 0% is effectively barren for that species.

In [ ]:
ar.plot_terrain_fill(run);
ar._final(ar.terrain_fill(run)).pivot(index='terrain', columns='species', values='fill')

## Where Folk spend their time, and their calories

In [ ]:
ar.plot_time_by_action(run);
ar.time_by_action(run).pivot(index='action', columns='decider', values='share')

In [ ]:
ar.plot_calories(run);
ar.calorie_ledger(run).pivot(index='category', columns='decider', values='kcal')

## Walking
Where Folk walk, how long a step takes there (1 is a tile per tick; slow ground and slopes raise it) and what a step costs in calories.

In [ ]:
ar.plot_movement(run);
ar.movement(run)

## Where food comes from
Calories by species and terrain for each decider, and how often attempts succeed.

In [ ]:
ar.plot_food_sources(run);
src = ar.food_sources(run)
src.groupby(['decider', 'species'])[['attempts', 'successes', 'kg', 'kcal']].sum()

## Per-Folk outcomes and decider parameters
Each Folk's counters with the parameter array it was born with. The correlations are a rough guide to which parameters matter (calories in and net calories per tick as a fitness proxy for a genetic algorithm), not proof.

In [ ]:
ar.plot_lifetimes(run);
life = ar.lifetimes(run)
life.groupby('decider')[['lived', 'meals', 'kcal_in_per_tick', 'kcal_out_per_tick', 'net_kcal_per_tick', 'injuries', 'goals', 'interrupts']].mean()

In [ ]:
ar.parameter_effects(run)

## Performance
Timing in microseconds: the whole tick, the ecology and Folk phases, the shared search, and each decider's `decide()`. Values at about 1.2 us are at the timer's resolution floor.

In [ ]:
ar.performance(run)

## Your own questions
Every table is a DataFrame; the raw files are also there:

```python
run.csv('entities.csv'), run.csv('lifetimes.csv'), run.events
```